# Rebuild Price Pressure Features

Самодостаточный ноутбук для пересборки `features/price_pressure` в основном проекте.

Причина пересборки: в старом агрегаторе `minute_bucket` считался как `timestamp.astype('int64') // 10**6`. В текущем pandas timestamp уже был в миллисекундах, поэтому bucket превращался из `1580601600000` в `1580601`, не совпадал с `klines.open_time`, merge давал пустые trade features, а затем всё заполнялось нулями.

Исправление: `minute_bucket = (transact_time // 60000) * 60000`, то есть тот же unix-ms начала минуты, что и `open_time` в klines.

In [ ]:
import io
import os
import re
from dataclasses import dataclass
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from dotenv import load_dotenv

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)
try:
    display
except NameError:
    display = print


## Settings

In [ ]:
@dataclass(frozen=True)
class Settings:
    s3_bucket: str = "binance-data-downloader"
    raw_prefix: str = "raw"
    features_prefix: str = "features"
    symbol: str = "ADAUSDT"
    interval: str = "1m"
    feature_dataset_name: str = "price_pressure"
    start_date: str | None = None
    end_date: str | None = None
    skip_existing: bool = False


settings = Settings(
    # Для полного восстановления оставь None..None.
    # Для точечной проверки можно поставить, например, "2020-02-01".."2020-02-03".
    start_date=None,
    end_date=None,
    # Старые price_pressure parquet сломаны, поэтому по умолчанию перезаписываем.
    skip_existing=False,
)

settings

In [ ]:
load_dotenv(dotenv_path=".env")

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("YC_ENDPOINT"),
    region_name=os.getenv("YC_REGION"),
    aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
)

missing_env = [name for name in ["YC_ENDPOINT", "YC_REGION", "YC_ACCESS_KEY_ID", "YC_SECRET_ACCESS_KEY"] if not os.getenv(name)]
if missing_env:
    raise ValueError(f"Missing .env values: {missing_env}")

## S3 Helpers

In [ ]:
FEATURE_COLUMNS = [
    "open_time",
    "VWAP",
    "PI_buy",
    "PI_sell",
    "PI_total",
    "F_eff",
    "F_PI",
    "F_asymmetry",
    "VWAP_pos",
]
FLOAT_FEATURE_COLUMNS = FEATURE_COLUMNS[1:]


def raw_klines_key(day, symbol=settings.symbol):
    return f"{settings.raw_prefix}/klines/symbol={symbol}/interval={settings.interval}/date={day}/data.parquet"


def raw_agg_trades_key(day, symbol=settings.symbol):
    return f"{settings.raw_prefix}/aggTrades/symbol={symbol}/date={day}/data.parquet"


def feature_dataset_key(day, symbol=settings.symbol):
    return f"{settings.features_prefix}/{settings.feature_dataset_name}/symbol={symbol}/interval={settings.interval}/date={day}/data.parquet"


def s3_key_exists(key):
    try:
        s3.head_object(Bucket=settings.s3_bucket, Key=key)
        return True
    except ClientError as exc:
        code = exc.response.get("Error", {}).get("Code")
        if code in {"404", "NoSuchKey", "NotFound"}:
            return False
        raise


def list_symbol_klines_days(symbol=settings.symbol):
    source_prefix = f"{settings.raw_prefix}/klines/symbol={symbol}/interval={settings.interval}/"
    pattern = re.compile(r"/date=(\d{4}-\d{2}-\d{2})/data\.parquet$")
    dates = set()
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=settings.s3_bucket, Prefix=source_prefix):
        for obj in page.get("Contents", []):
            match = pattern.search(f"/{obj['Key']}")
            if match:
                dates.add(match.group(1))
    return sorted(dates)




def list_symbol_agg_trades_days(symbol=settings.symbol):
    source_prefix = f"{settings.raw_prefix}/aggTrades/symbol={symbol}/"
    pattern = re.compile(r"/date=(\d{4}-\d{2}-\d{2})/data\.parquet$")
    dates = set()
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=settings.s3_bucket, Prefix=source_prefix):
        for obj in page.get("Contents", []):
            match = pattern.search(f"/{obj['Key']}")
            if match:
                dates.add(match.group(1))
    return sorted(dates)

def select_dates(available_dates):
    selected = []
    for day in available_dates:
        if settings.start_date and day < settings.start_date:
            continue
        if settings.end_date and day > settings.end_date:
            continue
        selected.append(day)
    return selected


def read_parquet_from_s3(key, columns=None):
    obj = s3.get_object(Bucket=settings.s3_bucket, Key=key)
    return pd.read_parquet(io.BytesIO(obj["Body"].read()), columns=columns)


def write_parquet_to_s3(df, key):
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow", compression="zstd")
    s3.put_object(Bucket=settings.s3_bucket, Key=key, Body=buffer.getvalue())

## Feature Algorithm

In [ ]:
def read_symbol_klines_day(day, symbol=settings.symbol):
    df = read_parquet_from_s3(raw_klines_key(day, symbol=symbol))
    required_columns = ["open_time", "high", "low", "close"]
    missing = sorted(set(required_columns) - set(df.columns))
    if missing:
        raise ValueError(f"Missing required klines columns: {missing}")
    if df.empty:
        return pd.DataFrame(columns=required_columns)

    cleaned = df[required_columns].copy()
    cleaned["open_time"] = pd.to_numeric(cleaned["open_time"], errors="coerce").astype("Int64")
    for column in ["high", "low", "close"]:
        cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce")

    cleaned = (
        cleaned.dropna(subset=["open_time"])
        .drop_duplicates(subset=["open_time"])
        .sort_values("open_time")
        .reset_index(drop=True)
    )

    open_time_utc = pd.to_datetime(cleaned["open_time"].astype("int64"), unit="ms", utc=True)
    if not open_time_utc.dt.second.eq(0).all() or not open_time_utc.dt.microsecond.eq(0).all():
        raise ValueError("open_time must be minute-aligned UTC")

    return cleaned


def read_symbol_agg_trades_day(day, symbol=settings.symbol):
    df = read_parquet_from_s3(raw_agg_trades_key(day, symbol=symbol))
    required_columns = ["transact_time", "price", "quantity", "is_buyer_maker"]
    missing = sorted(set(required_columns) - set(df.columns))
    if missing:
        raise ValueError(f"Missing required aggTrades columns: {missing}")
    if df.empty:
        return pd.DataFrame(columns=required_columns + ["minute_bucket"])

    cleaned = df[required_columns].copy()
    cleaned["transact_time"] = pd.to_numeric(cleaned["transact_time"], errors="coerce").astype("Int64")
    cleaned["price"] = pd.to_numeric(cleaned["price"], errors="coerce")
    cleaned["quantity"] = pd.to_numeric(cleaned["quantity"], errors="coerce")
    cleaned["is_buyer_maker"] = cleaned["is_buyer_maker"].astype("boolean")
    cleaned = cleaned.dropna(subset=required_columns)

    if cleaned.empty:
        return pd.DataFrame(columns=required_columns + ["minute_bucket"])

    # Correct unit: unix-ms start of minute, same scale as klines.open_time.
    cleaned["minute_bucket"] = (cleaned["transact_time"].astype("int64") // 60_000) * 60_000
    return cleaned.sort_values(["minute_bucket", "transact_time"], kind="stable").reset_index(drop=True)


def safe_divide(numerator, denominator):
    numerator_values = numerator.astype("float64").to_numpy()
    denominator_values = denominator.astype("float64").to_numpy()
    result = np.divide(
        numerator_values,
        denominator_values,
        out=np.zeros(len(numerator), dtype="float64"),
        where=denominator_values != 0,
    )
    return pd.Series(result, index=numerator.index)


def build_trade_features(agg_trades):
    if agg_trades.empty:
        return pd.DataFrame(columns=["open_time", "VWAP", "PI_buy", "PI_sell", "PI_total", "F_PI"])

    trades = agg_trades.copy()
    trades["quote_qty"] = trades["price"] * trades["quantity"]
    trades["is_buy"] = ~trades["is_buyer_maker"]
    trades["sign"] = np.where(trades["is_buy"], 1.0, -1.0)

    grouped = trades.groupby("minute_bucket", sort=True)
    minute_totals = grouped.agg(total_quantity=("quantity", "sum"), quote_qty_sum=("quote_qty", "sum"))
    minute_totals["VWAP"] = safe_divide(minute_totals["quote_qty_sum"], minute_totals["total_quantity"])

    trades = trades.join(minute_totals["VWAP"], on="minute_bucket")
    trades["delta_p"] = trades["price"] - trades["VWAP"]
    trades["PI_i"] = trades["sign"] * trades["quantity"] * trades["delta_p"]
    trades["price_diff_sq"] = grouped["price"].diff().pow(2).fillna(0.0)

    impact = trades.groupby(["minute_bucket", "is_buy"], sort=True)["PI_i"].sum().unstack(fill_value=0.0)
    pi_buy = impact[True] if True in impact.columns else pd.Series(0.0, index=impact.index)
    pi_sell = impact[False] if False in impact.columns else pd.Series(0.0, index=impact.index)
    rv = trades.groupby("minute_bucket", sort=True)["price_diff_sq"].sum()

    features = pd.DataFrame(index=minute_totals.index)
    features["open_time"] = features.index.astype("int64")
    features["VWAP"] = minute_totals["VWAP"]
    features["PI_buy"] = pi_buy.reindex(features.index, fill_value=0.0)
    features["PI_sell"] = pi_sell.reindex(features.index, fill_value=0.0)
    features["PI_total"] = features["PI_buy"] + features["PI_sell"]
    features["F_PI"] = safe_divide(features["PI_total"], rv * minute_totals["total_quantity"])
    return features.reset_index(drop=True)


def build_price_pressure_for_day(day, symbol=settings.symbol):
    backbone = read_symbol_klines_day(day, symbol=symbol)
    if backbone.empty:
        return pd.DataFrame(columns=FEATURE_COLUMNS)

    agg_trades = read_symbol_agg_trades_day(day, symbol=symbol)
    trade_features = build_trade_features(agg_trades)
    feature_df = backbone.merge(trade_features, on="open_time", how="left")

    if not agg_trades.empty and feature_df["VWAP"].notna().sum() == 0:
        raise ValueError("No aggTrades minutes matched klines open_time. Check minute_bucket/open_time units.")

    trade_value_columns = ["VWAP", "PI_buy", "PI_sell", "PI_total", "F_PI"]
    feature_df[trade_value_columns] = feature_df[trade_value_columns].fillna(0.0)

    buy_volume = agg_trades.loc[~agg_trades["is_buyer_maker"]].groupby("minute_bucket", sort=True)["quantity"].sum()
    sell_volume = agg_trades.loc[agg_trades["is_buyer_maker"]].groupby("minute_bucket", sort=True)["quantity"].sum()
    feature_df["V_buy_base"] = feature_df["open_time"].map(buy_volume).fillna(0.0)
    feature_df["V_sell_base"] = feature_df["open_time"].map(sell_volume).fillna(0.0)

    volume_imbalance_abs = (feature_df["V_buy_base"] - feature_df["V_sell_base"]).abs()
    feature_df["F_eff"] = safe_divide(feature_df["close"] - feature_df["VWAP"], volume_imbalance_abs)
    asymmetry_denominator = feature_df["PI_buy"] + feature_df["PI_sell"].abs()
    feature_df["F_asymmetry"] = safe_divide(feature_df["PI_buy"] - feature_df["PI_sell"].abs(), asymmetry_denominator)
    feature_df["VWAP_pos"] = safe_divide(feature_df["close"] - feature_df["VWAP"], feature_df["high"] - feature_df["low"])

    no_trade_mask = feature_df["V_buy_base"].eq(0) & feature_df["V_sell_base"].eq(0)
    feature_df.loc[no_trade_mask, FLOAT_FEATURE_COLUMNS] = 0.0
    feature_df = feature_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    feature_df = feature_df.sort_values("open_time").reset_index(drop=True)
    feature_df["open_time"] = feature_df["open_time"].astype("int64")
    for column in FLOAT_FEATURE_COLUMNS:
        feature_df[column] = feature_df[column].astype("float32")

    return feature_df[FEATURE_COLUMNS]

## Smoke Test

In [ ]:
test_day = "2020-02-02"
test_features = build_price_pressure_for_day(test_day)

print(test_day, test_features.shape)
display(test_features.head())
display(test_features.drop(columns=["open_time"]).agg(["min", "max", "mean", "std"]))
display(test_features.drop(columns=["open_time"]).nunique().sort_values())

constant_columns = [col for col in FLOAT_FEATURE_COLUMNS if test_features[col].nunique(dropna=False) <= 1]
if constant_columns:
    raise RuntimeError(f"Smoke test still has constant columns: {constant_columns}")

## Rebuild And Upload

In [ ]:
klines_dates = set(list_symbol_klines_days())
agg_trades_dates = set(list_symbol_agg_trades_days())
available_dates = sorted(klines_dates & agg_trades_dates)
skipped_no_agg_trades = sorted(klines_dates - agg_trades_dates)
selected_dates = select_dates(available_dates)

print(f"klines days: {len(klines_dates)}")
print(f"aggTrades days: {len(agg_trades_dates)}")
print(f"selected days with both raw sources: {len(selected_dates)}")
if skipped_no_agg_trades:
    print(f"skipped klines days without aggTrades: {skipped_no_agg_trades}")
print(selected_dates[:3], "...", selected_dates[-3:])


In [ ]:
rebuild_rows = []

for idx, day in enumerate(selected_dates, start=1):
    key = feature_dataset_key(day)
    if settings.skip_existing and s3_key_exists(key):
        rebuild_rows.append({"date": day, "rows": None, "key": key, "status": "skipped"})
        continue

    feature_df = build_price_pressure_for_day(day)
    constant_columns = [col for col in FLOAT_FEATURE_COLUMNS if feature_df[col].nunique(dropna=False) <= 1]
    if constant_columns:
        raise RuntimeError(f"{day}: constant price_pressure columns after rebuild: {constant_columns}")

    write_parquet_to_s3(feature_df, key)
    rebuild_rows.append({"date": day, "rows": len(feature_df), "key": key, "status": "uploaded"})

    if idx % 50 == 0 or idx == len(selected_dates):
        print(f"processed {idx}/{len(selected_dates)} days")

rebuild_results_df = pd.DataFrame(rebuild_rows)
rebuild_results_df

## Verify S3 Output

In [ ]:
verify_dates = [selected_dates[0], selected_dates[len(selected_dates) // 2], selected_dates[-1]] if selected_dates else []
verify_rows = []

for day in verify_dates:
    key = feature_dataset_key(day)
    df = read_parquet_from_s3(key)
    row = {"date": day, "rows": len(df), "key": key}
    for col in FLOAT_FEATURE_COLUMNS:
        row[f"{col}_nunique"] = int(df[col].nunique(dropna=False))
        row[f"{col}_min"] = float(df[col].min())
        row[f"{col}_max"] = float(df[col].max())
    verify_rows.append(row)

verify_df = pd.DataFrame(verify_rows)
verify_df

## Save Local Reports

In [ ]:
report_dir = Path("coverage_reports")
report_dir.mkdir(exist_ok=True)

if "rebuild_results_df" in globals():
    rebuild_results_df.to_csv(report_dir / "price_pressure_rebuild_results.csv", index=False)
if "verify_df" in globals():
    verify_df.to_csv(report_dir / "price_pressure_rebuild_verify.csv", index=False)

print(f"saved reports to {report_dir.resolve()}")